# PhysIQ Benchmark — Aggregate Analysis

This notebook collects results from the 5 task notebooks and computes the aggregate **PhysIQ Score**, **Format Robustness Score (FRS)**, and **Difficulty Scaling Score (DSS)**.

Run this notebook after all 5 task notebooks have been executed and their results saved to `../outputs/`.

## Scoring Summary

| Task | Weight | Instances |
|------|--------|----------|
| 1 – Trajectory Prediction | 20% | 60 × 3 fmt = 180 |
| 2 – Stability Judgment | 20% | 60 × 3 fmt = 180 |
| 3 – Causal Chain Reasoning | 20% | 60 × 3 fmt = 180 |
| 4 – Tool Use Planning | 20% | 40 × 3 fmt = 120 |
| 5 – Adaptive Replanning | 20% | 30 × 3 fmt = 90 |

**Format Robustness Score:** `FRS = 1 - (max - min) / max`  
**PhysIQ Score:** Equal-weighted average across all 5 tasks

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import sys, os, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Kaggle_agi_bench'))

import kaggle_benchmarks as kbench
print('kaggle-benchmarks', kbench.__version__)

OUTPUTS_DIR = '../outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Scoring utilities ────────────────────────────────────────────────
from physiq.scoring import (
    physiq_score,
    format_robustness_score,
    difficulty_scaling_score,
    is_significantly_different,
    TASK_WEIGHTS,
)

print('Task weights:', TASK_WEIGHTS)

In [ ]:
# ── Cell 3: Load per-task results (if saved as CSVs) ─────────────────────────
# Expected: each task notebook saves its df_results as a CSV.
# If not yet available, create empty placeholders.
task_files = {
    'trajectory':   f'{OUTPUTS_DIR}/task1_trajectory_results.csv',
    'stability':    f'{OUTPUTS_DIR}/task2_stability_results.csv',
    'causal_chain': f'{OUTPUTS_DIR}/task3_causal_chain_results.csv',
    'tool_use':     f'{OUTPUTS_DIR}/task4_tool_use_results.csv',
    'replanning':   f'{OUTPUTS_DIR}/task5_replanning_results.csv',
}

task_results = {}
for task_name, path in task_files.items():
    if os.path.exists(path):
        task_results[task_name] = pd.read_csv(path)
        print(f'Loaded {task_name}: {len(task_results[task_name])} rows')
    else:
        print(f'[MISSING] {task_name} → {path}')

print(f'\nLoaded {len(task_results)}/5 task result files.')

In [ ]:
# ── Cell 4: Aggregate PhysIQ Score ──────────────────────────────────────────
if len(task_results) == 5:
    task_means = {}
    for task_name, df in task_results.items():
        task_means[task_name] = df['score'].mean()
        print(f'  {task_name}: {task_means[task_name]:.3f}')

    physiq = physiq_score(task_means)
    print(f'\nPhysIQ Score: {physiq:.3f}')
else:
    print('Not all task results available yet. Run all 5 task notebooks first.')
    task_means = {t: 0.0 for t in TASK_WEIGHTS}

In [ ]:
# ── Cell 5: Format Robustness Score ─────────────────────────────────────────
if len(task_results) == 5:
    all_frs = []
    for task_name, df in task_results.items():
        for sid, grp in df.groupby('scenario_id'):
            scores = grp['score'].values
            mx = scores.max()
            frs = 1.0 - (mx - scores.min()) / mx if mx > 0 else 1.0
            all_frs.append(frs)

    mean_frs = float(np.mean(all_frs))
    print(f'Format Robustness Score (FRS): {mean_frs:.3f}')
    print('  1.0 = perfect consistency across JSON/ASCII/NL')
    print('  0.0 = completely inconsistent')

    # Per-task FRS
    print('\nPer-task FRS:')
    for task_name, df in task_results.items():
        frs_vals = []
        for sid, grp in df.groupby('scenario_id'):
            scores = grp['score'].values
            mx = scores.max()
            frs_vals.append(1.0 - (mx - scores.min()) / mx if mx > 0 else 1.0)
        print(f'  {task_name}: {np.mean(frs_vals):.3f}')

In [ ]:
# ── Cell 6: Difficulty Scaling Score ────────────────────────────────────────
if len(task_results) == 5:
    print('Difficulty scaling (should decrease easy → medium → hard):')
    for task_name, df in task_results.items():
        if 'difficulty' not in df.columns:
            continue
        d_means = df.groupby('difficulty')['score'].mean()
        easy = d_means.get('easy', 0)
        med  = d_means.get('medium', 0)
        hard = d_means.get('hard', 0)
        monotone = '✓' if easy >= med >= hard else '✗'
        print(f'  {task_name}: easy={easy:.3f} med={med:.3f} hard={hard:.3f} {monotone}')

In [ ]:
# ── Cell 7: Per-format breakdown ─────────────────────────────────────────────
if len(task_results) == 5:
    print('Score by representation format:')
    all_rows = pd.concat(list(task_results.values()), ignore_index=True)
    fmt_means = all_rows.groupby('representation')['score'].mean()
    for fmt, mean in fmt_means.items():
        print(f'  {fmt}: {mean:.3f}')

In [ ]:
# ── Cell 8: Visualizations ───────────────────────────────────────────────────
if len(task_results) == 5:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # ── Radar chart: per-task scores ──────────────────────────────────────────
    ax_radar = plt.subplot(2, 2, 1, polar=True)
    task_labels = list(task_means.keys())
    values = [task_means[t] for t in task_labels]
    n = len(task_labels)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    values += values[:1]
    angles += angles[:1]
    ax_radar.plot(angles, values, 'o-', linewidth=2, color='#2196F3')
    ax_radar.fill(angles, values, alpha=0.25, color='#2196F3')
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels([t.replace('_', ' ').title() for t in task_labels], size=9)
    ax_radar.set_ylim(0, 1)
    ax_radar.set_title('PhysIQ Per-Task Scores', size=11, pad=15)

    # ── Heatmap: task × difficulty ────────────────────────────────────────────
    ax_heat = axes[0, 1]
    heat_data = pd.DataFrame(index=['easy', 'medium', 'hard'])
    for task_name, df in task_results.items():
        if 'difficulty' in df.columns:
            d = df.groupby('difficulty')['score'].mean()
            heat_data[task_name[:8]] = [d.get('easy', 0), d.get('medium', 0), d.get('hard', 0)]
    im = ax_heat.imshow(heat_data.values, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn')
    ax_heat.set_xticks(range(len(heat_data.columns)))
    ax_heat.set_xticklabels(heat_data.columns, rotation=30, ha='right', size=8)
    ax_heat.set_yticks(range(3))
    ax_heat.set_yticklabels(['easy', 'medium', 'hard'])
    for i in range(3):
        for j in range(len(heat_data.columns)):
            ax_heat.text(j, i, f'{heat_data.values[i, j]:.2f}', ha='center', va='center', size=8)
    plt.colorbar(im, ax=ax_heat)
    ax_heat.set_title('Score by Task × Difficulty', size=11)

    # ── Bar: format robustness per task ───────────────────────────────────────
    ax_frs = axes[1, 0]
    task_frs = {}
    for task_name, df in task_results.items():
        frs_vals = []
        for sid, grp in df.groupby('scenario_id'):
            scores = grp['score'].values
            mx = scores.max()
            frs_vals.append(1.0 - (mx - scores.min()) / mx if mx > 0 else 1.0)
        task_frs[task_name[:8]] = float(np.mean(frs_vals))
    pd.Series(task_frs).plot(kind='bar', ax=ax_frs, color='#9C27B0')
    ax_frs.set_title('Format Robustness Score per Task', size=11)
    ax_frs.set_ylabel('FRS (1.0 = ideal)')
    ax_frs.set_ylim(0, 1)
    ax_frs.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    plt.setp(ax_frs.get_xticklabels(), rotation=30, ha='right')

    # ── Bar: per-format scores ────────────────────────────────────────────────
    ax_fmt = axes[1, 1]
    fmt_means.plot(kind='bar', ax=ax_fmt, color=['#2196F3', '#9C27B0', '#00BCD4'])
    ax_fmt.set_title('Overall Score by Representation Format', size=11)
    ax_fmt.set_ylabel('Mean Score (0-1)')
    ax_fmt.set_ylim(0, 1)
    plt.setp(ax_fmt.get_xticklabels(), rotation=0)

    plt.suptitle(f'PhysIQ Benchmark Results  (Score = {physiq:.3f})', size=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUTS_DIR}/physiq_aggregate_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {OUTPUTS_DIR}/physiq_aggregate_results.png')

In [ ]:
# ── Cell 9: Save summary CSV ─────────────────────────────────────────────────
if len(task_results) == 5:
    summary = pd.DataFrame([
        {
            'task': task_name,
            'mean_score': task_means[task_name],
            'weight': TASK_WEIGHTS.get(task_name, 0.2),
            'weighted': task_means[task_name] * TASK_WEIGHTS.get(task_name, 0.2),
        }
        for task_name in task_means
    ])
    summary.loc[len(summary)] = {
        'task': 'TOTAL',
        'mean_score': physiq,
        'weight': 1.0,
        'weighted': physiq,
    }
    summary.to_csv(f'{OUTPUTS_DIR}/physiq_summary.csv', index=False)
    print(summary.to_string(index=False))
    print(f'\nSaved → {OUTPUTS_DIR}/physiq_summary.csv')